## 데이터 분석 에이전트

이번 실습에서는 에이전트에게 공정 데이터를 조회하고 해당 데이터를 기반으로 질문에 답하는 에이전트를 만들어 보도록 하겠습니다.

데이터는 가상 ETCH 공정 데이터로, 2026년 8월 한 달 동안 처리한 Lot의 공정 조건과 검사 결과를 담고 있습니다. 에이전트는 질문에 답하기 위해 데이터베이스를 조회합니다.

In [2]:
import sqlite3
import pandas as pd

DB_PATH = "db/fab_etch.db"

# DB 연결
conn = sqlite3.connect(DB_PATH)

# DB 쿼리
query = """
SELECT *
FROM etch_lot
LIMIT 100
"""

# 쿼리 실행 및 데이터 df에 저장
df = pd.read_sql(query, conn)
# DB 연결 종료
conn.close()

print(df.shape)  # (100, 15)
df.head()

(100, 15)


,timestamp,lot_id,product_id,equipment_id,chamber_id,recipe_id,wafer_count,pressure_kpa,temperature_c,rf_power_w,gas_flow_sccm,process_time_s,defect_count,defect_rate,yield_rate
0,2026-08-01 22:42:00,L0801_01_A_A12_01,PROD_A12,ETCH_01,CH_A,RCP_A12,50,39.84,66.82,772.64,121.10,50.89,0,0.00,1.00
1,2026-08-01 06:24:00,L0801_01_A_A12_02,PROD_A12,ETCH_01,CH_A,RCP_A12,50,40.65,67.18,783.96,119.51,53.32,0,0.00,1.00
2,2026-08-01 06:20:00,L0801_01_A_A12_03,PROD_A12,ETCH_01,CH_A,RCP_A12,50,40.85,67.09,790.40,118.90,52.77,2,0.04,0.96
3,2026-08-01 23:35:00,L0801_01_A_A12_04,PROD_A12,ETCH_01,CH_A,RCP_A12,50,39.81,67.53,770.37,117.26,53.35,1,0.02,0.98
4,2026-08-01 03:02:00,L0801_01_A_A12_05,PROD_A12,ETCH_01,CH_A,RCP_A12,50,39.14,67.91,781.96,120.62,51.57,0,0.00,1.00


데이터는 아래와 같은 정보들을 포함하고 있습니다.

**[데이터 개요]**

- 파일: `fab_etch.db`
- 관측 단위: Lot 1건의 ETCH 공정 결과
- 기간: 2026-08-01 ~ 2026-08-31
- 행 수: 11,160건

**[컬럼 설명]**

| 컬럼 | 설명 | 단위/형식 |
|---|---|---|
| `timestamp` | 공정 완료 시각 | YYYY-MM-DD HH:MM:SS |
| `lot_id` | Lot 식별자 | 문자열 |
| `product_id` | 제품군 코드 | 범주형 |
| `equipment_id` | 설비 ID | ETCH_01~03 |
| `chamber_id` | 설비 내 챔버 ID | CH_A~C |
| `recipe_id` | 공정 Recipe ID | 4종 |
| `wafer_count` | Lot 내 처리 Wafer 수 | 개 |
| `pressure_kpa` | 압력 센서 기록값 | kPa |
| `temperature_c` | 공정 온도 기록값 | °C |
| `rf_power_w` | RF Power 기록값 | W |
| `gas_flow_sccm` | 가스 유량 기록값 | sccm |
| `process_time_s` | 공정 시간 | 초 |
| `defect_count` | 검사에서 집계된 불량 Wafer 수 | 개 |
| `defect_rate` | `defect_count / wafer_count` | 비율(0~1) |
| `yield_rate` | `1 - defect_rate` | 비율(0~1) |

**실습** : 한번 continue에게 SQL 쿼리를 요청해서 원하는 데이터를 직접 조회해 봅시다.

In [ ]:
# DB 연결
conn = sqlite3.connect(DB_PATH)

# DB 쿼리
query = """

"""

# 쿼리 실행 및 데이터 df에 저장
df = pd.read_sql(query, conn)
# DB 연결 종료
conn.close()

print(df.shape)
df.head()

파이썬으로 db 쿼리 조회가 가능하단 것을 확인했으니, 이 코드를 활용할 수 있는 툴을 만들어 에이전트에 제공해 보도록 하겠습니다.

우선 툴로 만들어 볼까요?

In [4]:
from langchain.tools import tool

@tool
def run_sql(query: str) -> str:
    """DB에 SQL 쿼리를 실행하고 결과를 돌려줍니다."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(query, conn)  # 에이전트가 작성한 쿼리를 실행
    conn.close()
    return df.to_string(index=False)  # 조회 결과를 문자열로 바꿔서 에이전트에게 전달

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


그 뒤에 툴을 이용하는 에이전트를 구현합니다.

In [5]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain.agents import create_agent

from langchain_openai import ChatOpenAI

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(find_dotenv())

model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=os.getenv("gpt_4o_mini_BASE_URL"),
    temperature=0,  # 같은 질문에 최대한 같은 답을 하도록 설정
)

agent = create_agent(
    model,
    tools=[run_sql],
    system_prompt="너는 반도체 공정 데이터를 분석하는 어시스턴트다. 데이터에 대한 질문은 run_sql 도구로 DB를 조회해서 답해라.",
)

이제 에이전트에게 질문을 해 보겠습니다.

In [ ]:
messages = [
    {"role": "user", "content": "설비별 평균 온도를 알려줘"},
]

response = agent.invoke({"messages": messages})

print("답변 :\n", response['messages'][-1].content)

DatabaseError: Execution failed on sql 'SELECT equipment_id, AVG(temperature) as average_temperature FROM temperature_data GROUP BY equipment_id;': no such table: temperature_data

문제가 발생한 이유는 에이전트가 DB를 직접 볼 수 없기 때문입니다.

에이전트가 데이터에 대해 알 수 있는 정보는 "도구의 설명(docstring)이 전부" 입니다. 설명에 "DB에 SQL 쿼리를 실행한다"라고만 적혀 있으니, 테이블과 컬럼 이름을 추측해서 쿼리를 작성한 것입니다.

In [7]:
@tool
def run_sql(query: str) -> str:
    """ETCH 공정 DB(SQLite)에 SQL 쿼리를 실행하고 결과를 돌려줍니다.

    테이블: etch_lot (한 행 = Lot 1건의 ETCH 공정 결과)
    컬럼:
    - equipment_id  : 설비 ID (ETCH_01, ETCH_02, ETCH_03)
    - chamber_id    : 설비 내 챔버 ID (CH_A, CH_B, CH_C)
    - temperature_c : 공정 온도 기록값 (°C)
    """
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql(query, conn)
    conn.close()
    return df.to_string(index=False)

# 도구를 수정했으니 에이전트도 다시 만들어 줍니다.
agent = create_agent(
    model,
    tools=[run_sql],
    system_prompt="너는 반도체 공정 데이터를 분석하는 어시스턴트다. 데이터에 대한 질문은 run_sql 도구로 DB를 조회해서 답해라.",
)


In [8]:
messages = [
    {"role": "user", "content": "설비별 평균 온도를 알려줘"},
]

response = agent.invoke({"messages": messages})

print("답변 :\n", response['messages'][-1].content)

답변 :
 설비별 평균 온도는 다음과 같습니다:

- ETCH_01: 69.36 °C
- ETCH_02: 69.57 °C
- ETCH_03: 69.28 °C


이제 제대로 답변하는 것을 볼 수 있습니다! 이처럼 docstring 또는 시스템 프롬프트에 데이터에 관한 자세한 설명이 있어야 llm이 알맞은 쿼리를 작성할 수 있다는 것을 확인할 수 있습니다.

현재 도구 설명에는 컬럼 3개만 적었습니다. 만약 설명에 없는 정보를 물어보면 어떻게 될까요?

In [9]:
messages = [
    {"role": "user", "content": "설비별 불량률을 알려줘"},
]

response = agent.invoke({"messages": messages})

print("답변 :\n", response['messages'][-1].content)

DatabaseError: Execution failed on sql 'SELECT equipment_id, COUNT(*) AS total_lots, SUM(CASE WHEN defect = 1 THEN 1 ELSE 0 END) AS defect_lots, (SUM(CASE WHEN defect = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS defect_rate FROM etch_lot GROUP BY equipment_id;': no such column: defect

이번에도 마찬가지로 같은 에러가 발생합니다.

문제는 이렇게 에러가 발생하면 챗봇 서비스 자체가 정지된다는 것입니다. 챗봇은 답변을 생성하다가 멈추게 되고, 사용자 입장에선 알 수 없는 에러 메세지만 받게 되는 것이죠.

그러면 어떻게 이 문제를 해결해야 할까요? 발생할 수 있는 모든 에러 상황에 대해서 시스템 프롬프트에 작성하면 될까요?

저희는 그보다 더 영리한 방법을 알고 있는 것 같습니다. 바로 LLM은 추론 능력을 갖고 있다는 것이죠.

지금은 에러가 발생하면 툴 함수가 그대로 멈추면서 답변도 종료됩니다. 따라서 이런 코드 실행 툴은 항상 에러가 발생할 수 있다는 것을 인지하고, 에러가 발생했을 때 에러 메시지를 반환하도록 하여 llm이 적절히 대응할 수 있도록 하는 방법을 사용합니다.

In [10]:
@tool
def run_sql(query: str) -> str:
    """ETCH 공정 DB(SQLite)에 SQL 쿼리를 실행하고 결과를 돌려줍니다.

    테이블: etch_lot (한 행 = Lot 1건의 ETCH 공정 결과)
    컬럼:
    - equipment_id  : 설비 ID (ETCH_01, ETCH_02, ETCH_03)
    - chamber_id    : 설비 내 챔버 ID (CH_A, CH_B, CH_C)
    - temperature_c : 공정 온도 기록값 (°C)
    """
    conn = sqlite3.connect(DB_PATH)
    try:
        df = pd.read_sql(query, conn)
    except Exception as e:
        # 에러가 발생하면 멈추지 않고, 에러 메시지를 에이전트에게 돌려줍니다.
        return f"쿼리 실행 오류: {e}"
    finally:
        conn.close()  # 에러가 나더라도 DB 연결은 항상 종료
    return df.to_string(index=False)

agent = create_agent(
    model,
    tools=[run_sql],
    system_prompt="너는 반도체 공정 데이터를 분석하는 어시스턴트다. 데이터에 대한 질문은 run_sql 도구로 DB를 조회해서 답해라.",
)

messages = [
    {"role": "user", "content": "설비별 불량률을 알려줘"},
]

response = agent.invoke({"messages": messages})

print("답변 :\n", response['messages'][-1].content)


답변 :
 쿼리를 실행하는 과정에서 "defect"라는 컬럼이 없다는 오류가 발생했습니다. 불량률을 계산하기 위해 필요한 데이터가 없거나 다른 방식으로 접근해야 할 것 같습니다. 

불량률을 계산하기 위해 어떤 기준으로 불량을 정의할 수 있는지, 또는 다른 관련 정보를 제공해 주실 수 있나요?


이제 챗봇이 멈추지 않고 답변하는 것을 확인할 수 있습니다. 하지만 챗봇의 특성상 결과가 매번 다르게 나올 수 있습니다.

테이블 구조를 스스로 조회해서 'defect_rate' 컬럼을 찾아낸 뒤 정확하게 답할 수도 있고, "온도가 100도를 넘으면 불량이라고 판정"처럼 **데이터에 없는 기준을 스스로 만들어서** 틀린 답을 하기도 합니다.

특히 두 번째의 경우 에러도 발생하지 않고 답변도 그럴듯하기 때문에, 답변만 봐서는 틀렸다는 것을 파악하기 어렵습니다.

모든 llm의 에러를 통제하는 것은 지금 단계에서 어려울 수 있습니다. 대신 **llm이 어떤 에러를 발생시켰는지 언제든 추적할 수 있도록 llm의 수행 내역을 출력**하여 언제든 확인할 수 있도록 하는 것이 중요합니다.

에이전트가 에러를 받은 뒤 어떻게 대응했는지 확인하기 위해, 도구 호출 기록을 출력하는 함수를 만들어 봅시다.

`response['messages']`에는 사용자 질문, 에이전트의 도구 호출(실행한 쿼리), 도구의 반환값(조회 결과), 최종 답변이 순서대로 기록되어 있습니다.

In [11]:
# 에이전트가 실행한 쿼리와 도구가 돌려준 결과를 순서대로 출력
for message in response['messages']:
    if message.type == "ai":
        for tool_call in message.tool_calls:
            print("[실행한 쿼리]\n", tool_call['args']['query'])
    elif message.type == "tool":
        print("[조회 결과]\n", message.content[:500], "\n")  # 결과가 길면 앞부분만 출력


[실행한 쿼리]
 SELECT equipment_id, COUNT(*) AS total_lots, SUM(CASE WHEN defect = 1 THEN 1 ELSE 0 END) AS defect_lots FROM etch_lot GROUP BY equipment_id;
[실행한 쿼리]
 SELECT equipment_id, COUNT(*) AS total_lots, SUM(CASE WHEN defect = 1 THEN 1 ELSE 0 END) AS defect_lots FROM etch_lot GROUP BY equipment_id;
[실행한 쿼리]
 SELECT equipment_id, COUNT(*) AS total_lots, SUM(CASE WHEN defect = 1 THEN 1 ELSE 0 END) AS defect_lots FROM etch_lot GROUP BY equipment_id;
[조회 결과]
 쿼리 실행 오류: Execution failed on sql 'SELECT equipment_id, COUNT(*) AS total_lots, SUM(CASE WHEN defect = 1 THEN 1 ELSE 0 END) AS defect_lots FROM etch_lot GROUP BY equipment_id;': no such column: defect 

[조회 결과]
 쿼리 실행 오류: Execution failed on sql 'SELECT equipment_id, COUNT(*) AS total_lots, SUM(CASE WHEN defect = 1 THEN 1 ELSE 0 END) AS defect_lots FROM etch_lot GROUP BY equipment_id;': no such column: defect 

[조회 결과]
 쿼리 실행 오류: Execution failed on sql 'SELECT equipment_id, COUNT(*) AS total_lots, SUM(CASE WHEN defect = 1 THEN 1 EL

#### 정리 : 데이터 조회 에이전트의 문제점과 대응 방향

위에서 확인한 문제는 두 가지입니다.

1. 같은 질문이라도 실행할 때마다 결과가 달라질 수 있다.
2. 에러 없이 그럴듯한 답변이 나와도, 그 답변이 틀렸을 수 있다.

두 문제의 공통 원인은 **에이전트가 모르는 부분을 추측으로 채운다**는 점입니다. 도구 설명에 적어 둔 온도 컬럼은 정확하게 조회했지만, 설명에 없던 불량률은 컬럼 이름이나 계산 기준을 추측했습니다.

**[대응 방향]**

| 방향 | 내용 | 수정 위치 |
|---|---|---|
| 정보 제공 | 에이전트가 필요한 테이블·컬럼 정보를 빠짐없이 전달 | 도구 설명 (docstring) |
| 규칙 설정 | 모르는 정보나 데이터에 없는 내용을 어떻게 처리할지 지정 | 시스템 프롬프트 |
| 동작 제한 | 에러 메시지 반환, 조회 결과 행 수 제한 등 실제 실행 방식 지정 | 도구 코드 |

- 조회 결과 행 수 제한 : 결과가 수천 행이면 모델이 한 번에 읽을 수 있는 양(컨텍스트)을 넘거나 비용이 크게 늘어납니다.

**[결과 확인 방법]**

- 답변만 보지 않고, 에이전트가 실행한 쿼리와 조회 결과를 함께 확인합니다.
- 결과가 실행마다 달라질 수 있으므로, 같은 질문을 여러 번 실행해서 기준 답변과 비교합니다.
- 에이전트의 답변은 1차 분석 결과로 활용하고, 최종 판단은 근거를 확인한 뒤 사람이 내립니다.

실습에서는 위 세 가지 수정 위치를 활용해 에이전트를 직접 보강해 봅니다.


### 데이터 조회 에이전트 구축하기

sqlite3 라이브러리를 활용해 직접 db로부터 데이터를 조회하고 분석할 수 있는 에이전트를 구축해 봅니다.

구축된 에이전트가 데이터를 잘 불러오는지 확인하고, 문제가 있다면 어떤 문제가 있는지 직접 확인해 보고 해결 방법을 강구해 봅니다.

* **실습안내** : `03_데이터분석에이전트_실습안내.md`
* **실습폴더** : `03_Data_fetch_agent`